## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [0]:
import pandas as pd

In [204]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
data = pd.read_csv('/content/drive/My Drive/lab_residency6/prices.csv')

### Check all columns in the dataset

In [206]:
data.columns

Index(['date', 'symbol', 'open', 'close', 'low', 'high', 'volume'], dtype='object')

### Drop columns `date` and  `symbol`

In [0]:
data = data.drop(['date', 'symbol'], axis=1)

In [208]:
data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


In [209]:
data.shape

(851264, 5)

### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [0]:
data = data.loc[:999, :]

In [0]:
data['volume'] = data['volume']/1000000

In [212]:
data.shape

(1000, 5)

### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
X = data[['open', 'close',	'low',	'high']]

In [0]:
y= data['volume']
y=y.astype('float32')

In [0]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)

#### Convert Training and Test Data to numpy float32 arrays


In [0]:
x_train=x_train.astype('float32')

In [0]:
x_test=x_test.astype('float32')

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn import preprocessing

In [0]:
#Input features
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')

#Normalize the data
x_n = preprocessing.normalize(x_train)
x_n = x_n.astype('float32')

#Actual Prices
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')


## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
import numpy as np

In [0]:
#W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
W = tf.Variable(tf.random_normal([4, 1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

In [0]:
#We will use normalized data
#y = tf.add(tf.matmul(x,W),b,name='output')
y = tf.add(tf.matmul(x_n,W),b,name='output')

2.Define a function to calculate prediction

In [0]:
loss = tf.reduce_mean(tf.square(y-y_),name='Loss')

3.Loss (Cost) Function [Mean square error]

In [0]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [228]:
for epoch in range(training_epochs):
            
    #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:x_train, y_:y_train})
    
    if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', train_loss)

Training loss at step:  0  is  301.96658
Training loss at step:  5  is  281.75595
Training loss at step:  10  is  276.12704
Training loss at step:  15  is  274.55936
Training loss at step:  20  is  274.12277
Training loss at step:  25  is  274.00113
Training loss at step:  30  is  273.9673
Training loss at step:  35  is  273.95786
Training loss at step:  40  is  273.9552
Training loss at step:  45  is  273.95447
Training loss at step:  50  is  273.95428
Training loss at step:  55  is  273.95422
Training loss at step:  60  is  273.9542
Training loss at step:  65  is  273.9542
Training loss at step:  70  is  273.95422
Training loss at step:  75  is  273.95422
Training loss at step:  80  is  273.9542
Training loss at step:  85  is  273.95422
Training loss at step:  90  is  273.9542
Training loss at step:  95  is  273.95422


### Get the shapes and values of W and b

In [229]:
W.shape
W

<tf.Variable 'Weights_12:0' shape=(4, 1) dtype=float32_ref>

In [230]:
b.shape

TensorShape([Dimension(1)])

### Model Prediction on 1st Examples in Test Dataset

In [0]:
feed_dict = {x: x_test}  
y_pred = y.eval(feed_dict, session=sess) 

In [236]:
y_pred

array([[5.6103826],
       [5.607754 ],
       [5.608819 ],
       [5.618808 ],
       [5.616083 ],
       [5.6150465],
       [5.613774 ],
       [5.6032124],
       [5.6149626],
       [5.612602 ],
       [5.6164722],
       [5.6020536],
       [5.614557 ],
       [5.6083384],
       [5.6197023],
       [5.6030836],
       [5.6123414],
       [5.6150866],
       [5.6109657],
       [5.6022654],
       [5.606207 ],
       [5.610275 ],
       [5.6189566],
       [5.6158085],
       [5.599986 ],
       [5.615213 ],
       [5.61309  ],
       [5.6063538],
       [5.6137743],
       [5.6131163],
       [5.609998 ],
       [5.5836544],
       [5.6043935],
       [5.6110606],
       [5.5876923],
       [5.5931168],
       [5.6112003],
       [5.6141014],
       [5.6122665],
       [5.6010303],
       [5.6094027],
       [5.6113577],
       [5.608717 ],
       [5.614362 ],
       [5.6187763],
       [5.596734 ],
       [5.6091537],
       [5.614899 ],
       [5.6033554],
       [5.61644  ],


None


## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [0]:
data_iris = pd.read_csv('/content/drive/My Drive/lab_residency6/11_Iris.csv')

In [263]:
data_iris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [0]:
data_iris= data_iris.drop(['Id'], axis=1)

In [0]:
data_iris= pd.get_dummies(data=data_iris, columns=['Species'])

In [266]:
data_iris.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
0,5.1,3.5,1.4,0.2,1,0,0
1,4.9,3.0,1.4,0.2,1,0,0
2,4.7,3.2,1.3,0.2,1,0,0
3,4.6,3.1,1.5,0.2,1,0,0
4,5.0,3.6,1.4,0.2,1,0,0


### Splitting the data into feature set and target set

In [0]:
X = data_iris[['SepalLengthCm',	'SepalWidthCm',	'PetalLengthCm',	'PetalWidthCm']]

In [0]:
y = data_iris[['Species_Iris-setosa', 'Species_Iris-versicolor',	'Species_Iris-virginica', ]]

In [0]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [0]:
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers.core import Dense,Dropout,Activation,Flatten
from keras.layers.convolutional import Conv2D,MaxPooling2D
from keras.utils import np_utils

In [0]:
# create model
model = Sequential()
model.add(Dense(12,input_dim=4, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(3, activation='softmax'))
# Compile model
model.compile(loss='binary_crossentropy', optimizer='sgd', metrics=['accuracy'])


### Model Training 

In [279]:
# Fit the model
model.fit(x_train, y_train, validation_data=(x_test,y_test), epochs=10, batch_size=5)

Train on 105 samples, validate on 45 samples
Epoch 1/10
105/105 [==============================] - 0s 3ms/step - loss: 0.7558 - acc: 0.7302 - val_loss: 0.4726 - val_acc: 0.7556
Epoch 2/10
105/105 [==============================] - 0s 894us/step - loss: 0.4292 - acc: 0.8381 - val_loss: 0.4190 - val_acc: 0.8741
Epoch 3/10
105/105 [==============================] - 0s 774us/step - loss: 0.3890 - acc: 0.8825 - val_loss: 0.3930 - val_acc: 0.8444
Epoch 4/10
105/105 [==============================] - 0s 795us/step - loss: 0.3608 - acc: 0.8667 - val_loss: 0.3668 - val_acc: 0.8444
Epoch 5/10
105/105 [==============================] - 0s 838us/step - loss: 0.3375 - acc: 0.8730 - val_loss: 0.3503 - val_acc: 0.8593
Epoch 6/10
105/105 [==============================] - 0s 821us/step - loss: 0.3156 - acc: 0.8825 - val_loss: 0.3371 - val_acc: 0.8148
Epoch 7/10
105/105 [==============================] - 0s 805us/step - loss: 0.3026 - acc: 0.8730 - val_loss: 0.3100 - val_acc: 0.8741
Epoch 8/10
105/105 

### Model Prediction

In [0]:
y_predict=model.predict_classes(x_test)

In [281]:
y_predict

array([2, 2, 0, 2, 2, 0, 1, 1, 0, 1, 2, 1, 0, 2, 0, 2, 2, 2, 0, 0, 1, 2,
       2, 2, 2, 2, 1, 1, 2, 2, 2, 1, 0, 2, 1, 0, 0, 0, 0, 2, 2, 1, 2, 2,
       1])

### Save the Model

In [0]:
from keras.models import load_model

model.save('/content/drive/My Drive/lab_residency6/iris_pred_keras.h5')

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?